In [2]:
import os
import pandas as pd

BASE_DIR = os.getcwd()
DATA_DIR = os.path.normpath(os.path.join(BASE_DIR, "..", "data"))
TAXONOMY_PATH = os.path.join(DATA_DIR, "skill_taxonomy.csv")
TRAINING_PATH = os.path.join(DATA_DIR, "training_data.csv")

tax = pd.read_csv(TAXONOMY_PATH)

rows = []

for _, row in tax.iterrows():
    skill = row["skill"]
    category = row["category"]

    # Add canonical skill name itself
    rows.append((skill, category))

    # Add Each Alias
    if isinstance(row["aliases"], str) and row["aliases"].strip():
        for alias in row["aliases"].split(";"):
            rows.append((alias.strip(), category))

training_df = pd.DataFrame(rows, columns=["text", "label"])
training_df.to_csv(TRAINING_PATH, index=False)

print(f"Built {len(training_df)} training examples across {training_df['label'].nunique()} categories")
print(training_df["label"].value_counts())
print(training_df.head(10))

Built 113 training examples across 9 categories
label
Data Analytics          27
Programming             23
Database                18
Machine Learning        12
DevOps                  12
Software Engineering     7
Cloud                    6
Business                 6
Data Engineering         2
Name: count, dtype: int64
             text        label
0          Python  Programming
1         python3  Programming
2        python 3  Programming
3              py  Programming
4            Java  Programming
5         java se  Programming
6      JavaScript  Programming
7              js  Programming
8  javascript es6  Programming
9            HTML  Programming


In [3]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

MODEL_DIR = os.path.join(BASE_DIR, "..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, "skill_classifier.pkl")

df = pd.read_csv(TRAINING_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

joblib.dump(pipeline, MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")

                      precision    recall  f1-score   support

            Business       0.00      0.00      0.00         1
               Cloud       0.00      0.00      0.00         2
      Data Analytics       0.22      1.00      0.36         4
            Database       1.00      0.67      0.80         3
              DevOps       1.00      0.50      0.67         2
    Machine Learning       0.00      0.00      0.00         4
         Programming       1.00      0.33      0.50         6
Software Engineering       0.00      0.00      0.00         1

            accuracy                           0.39        23
           macro avg       0.40      0.31      0.29        23
        weighted avg       0.52      0.39      0.36        23

Saved model to c:\Users\Admin\Desktop\Internship Task\Job Skill extraction\Job_Skill_Extraction\notebooks\..\models\skill_classifier.pkl


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [4]:
loaded_model = joblib.load(MODEL_PATH)

new_terms = ["Kubernetes", "MongoDB", "Vue.js", "Excel", "Docker", "Tableau"]
predictions = loaded_model.predict(new_terms)

for term, label in zip(new_terms, predictions):
    print(f"{term} -> {label}")

Kubernetes -> DevOps
MongoDB -> Database
Vue.js -> Programming
Excel -> Data Analytics
Docker -> DevOps
Tableau -> Data Analytics


## Summary

I converted my skill taxonomy and its aliases into labeled training examples, used TF-IDF to convert skill names into numerical features, trained Logistic Regression to classify skills into categories, evaluated it on test data, and saved the complete pipeline using Joblib for future predictions.